In [ ]:
import sqlite3


In [ ]:
conn = sqlite3.connect("my_datawarehouse.db")
cursor = conn.cursor()


In [ ]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Dim_Drug(
    drugId INTEGER PRIMARY KEY,
    drugName VARCHAR(255)
    forDisease VARCHAR(255)
);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Dim_HealthUnit(
    healthUnitId INTEGER PRIMARY KEY,
    healthUnitName VARCHAR(255),
    address VARCHAR(255),
    districtId INTEGER,
);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Dim_Vaccine(
    vaccineId INTEGER PRIMARY KEY,
    vaccineName VARCHAR(255),
);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Dim_Quarantine(
    quarantineId INTEGER PRIMARY KEY,
    quarantineName VARCHAR(255),
    numberofDays INTEGER,
);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Dim_Venue(
  venueId INTEGER PRIMARY KEY,
  venueType VARCHAR,
  venueAddress VARCHAR,
);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Dim_Time(
  dateId INTEGER PRIMARY KEY,
  date DATE,
  yearId INTEGER,
  MonthId INTEGER,
  WeekId INTEGER,
  dayOfWeekId INTEGER,

);
""")


cursor.execute("""CREATE TABLE IF NOT EXISTS Fact_Patient_Drug (
    patientId INTEGER,
    drugId INTEGER,
    startDateId INTEGER,
    endDateId INTEGER,
    Dose INTEGER,
    PRIMARY KEY (patientId, drugId, startDateId, endDateId),
    FOREIGN KEY (patientId) REFERENCES Fact_Patient (patientId),
    FOREIGN KEY (drugId) REFERENCES Dim_Drug (drugId),
    FOREIGN KEY (startDateId) REFERENCES Dim_Time (dateId),
    FOREIGN KEY (endDateId) REFERENCES Dim_Time (dateId)

);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Fact_Patient(
  patientId INTEGER,
  diseaseVariantId INTEGER,
  dateId INTEGER,
  patientStatusId INTEGER,
  PRIMARY KEY (patientId, diseaseVariantId, dateId, patientStatusId),

);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Fact_Citizen_Vaccine (
    citizenId INTEGER,
    dateId INTEGER,
    healthUnitId INTEGER,
    vaccineId INTEGER,
    PRIMARY KEY (citizenId, dateId, healthUnitId, vaccineId),
    FOREIGN KEY (citizenId) REFERENCES Fact_Citizen_Quarantine(citizenId),
    FOREIGN KEY (dateId) REFERENCES Dim_Time (dateId),
    FOREIGN KEY (healthUnitId) REFERENCES Dim_HealthUnit (healthUnitId)
);
""")


cursor.execute("""CREATE TABLE IF NOT EXISTS Fact_Citizen_Quarantine (
    citizenId INTEGER,
    quarantineId INTEGER,
    startDateId INTEGER,
    endDateId INTEGER,
    PRIMARY KEY (citizenId, quarantineId, startDateId, endDateId),
    FOREIGN KEY (citizenId) REFERENCES Fact_Citizen_Venue (citizenId),
    FOREIGN KEY (quarantineId) REFERENCES Dim_Quarantine (quarantineId),
    FOREIGN KEY (startDateId) REFERENCES Dim_Time (dateId),
    FOREIGN KEY (endDateId) REFERENCES Dim_Time (dateId)

);
""")

cursor.execute("""CREATE TABLE IF NOT EXISTS Fact_Citizen_Venue (
    citizenId INTEGER,
    venueId INTEGER,
    dateId INTEGER,
    PRIMARY KEY (citizenId, venueId, dateId),
    FOREIGN KEY (dateId) REFERENCES Dim_Time (dateId),
    FOREIGN KEY (venueId) REFERENCES Dim_Venue (venueId),

);
""")

conn.commit()


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS MV_Q1 AS
SELECT
    t.yearId, t.MonthId, v.venueId,
    COUNT(DISTINCT p.patientId) AS total_cases,
    COUNT(DISTINCT cv.citizenId) AS total_vaccinations,
    COUNT(DISTINCT cq.citizenId) AS total_quarantines,
    COUNT(DISTINCT pd.patientId) AS total_treatments
FROM Fact_Citizen_Venue v
LEFT JOIN Fact_Patient p ON v.citizenId = p.patientId
LEFT JOIN Fact_Citizen_Vaccine cv ON v.citizenId = cv.citizenId
LEFT JOIN Fact_Citizen_Quarantine cq ON v.citizenId = cq.citizenId
LEFT JOIN Fact_Patient_Drug pd ON p.patientId = pd.patientId
JOIN Dim_Time t ON v.dateId = t.dateId
GROUP BY t.yearId, t.MonthId, v.venueId;
""")
conn.commit()


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS MV_Q2 AS
SELECT
    t.yearId, t.MonthId,
    COUNT(DISTINCT p.patientId) AS total_cases,
    COUNT(DISTINCT cq.citizenId) AS total_quarantines,
    COUNT(DISTINCT pd.patientId) AS total_treatments
FROM Fact_Patient p
LEFT JOIN Fact_Citizen_Quarantine cq ON p.patientId = cq.citizenId
LEFT JOIN Fact_Patient_Drug pd ON p.patientId = pd.patientId
JOIN Dim_Time t ON p.dateId = t.dateId
GROUP BY t.yearId, t.MonthId;
""")
conn.commit()
